# 문제해결 1: 금융사기 보고서 기반 FAISS Naive RAG

## 실습 목적

Naive RAG의 기본 흐름을 익힌 후, **직접 문서 기반 질의응답 시스템을 구성하고 문제를 점검**하도록 설계한 실습 예제

사용 문서: `data/진화하는 금융사기, 모두의 경계가 필요한 시점.pdf`

## 실습 시나리오

금융소비자 교육 담당자가 하나의 금융사기 보고서를 기반으로 질문에 답변하는 내부용 QA 챗봇을 만들려고 한다.  

## 완성 기준

| 기준 | 설명 |
|---|---|
| 문서 로딩 | PDF 페이지를 `Document` 객체로 정상 로딩한다. |
| 청킹 | 문서가 적절한 크기의 청크로 분할된다. |
| 벡터DB 생성 | FAISS 벡터스토어가 생성되고 로컬 저장까지 가능하다. |
| 검색 | 질문과 관련된 상위 문서 조각을 검색한다. |
| 답변 생성 | 검색된 문서에 근거해서만 답변한다. |

## 0. 설치

> `faiss-cpu`는 로컬 CPU 환경에서 FAISS를 사용하기 위한 패키지이다.

In [1]:
# uv add langchain langchain-openai langchain-community langchain-text-splitters pypdf faiss-cpu python-dotenv pandas

## 1. 라이브러리 불러오기

LangChain의 기본 구성요소를 사용한다.

- `PyPDFLoader`: PDF 문서 로딩
- `RecursiveCharacterTextSplitter`: 문서 청킹
- `OpenAIEmbeddings`: 텍스트 임베딩
- `FAISS`: 로컬 벡터스토어
- `ChatOpenAI`: 답변 생성 LLM

In [2]:
from pathlib import Path
import os
from pprint import pprint

import pandas as pd
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

/tmp/ipykernel_99268/1765998713.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## 2. 환경변수 설정

OpenAI API Key는 `.env` 파일 또는 운영체제 환경변수에 설정한다.

`.env` 파일 예시는 다음과 같다.

```text
OPENAI_API_KEY=sk-...
```

In [3]:
load_dotenv(override=True, dotenv_path="../.env")

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(
        "OPENAI_API_KEY가 설정되어 있지 않습니다. .env 파일 또는 환경변수에 OPENAI_API_KEY를 설정하세요."
    )

print("OPENAI_API_KEY 설정 확인 완료")

OPENAI_API_KEY 설정 확인 완료


## 3. PDF 파일 경로 설정

노트북과 같은 폴더 또는 `data` 폴더에 PDF를 넣어두면 자동으로 찾는다.

권장 폴더 구조는 다음과 같다.

```text
project/
├─ 1_faiss_naive_rag_financial_fraud.ipynb
└─ data/
   └─ 진화하는 금융사기, 모두의 경계가 필요한 시점.pdf
```

In [4]:
PDF_NAME = "진화하는 금융사기, 모두의 경계가 필요한 시점.pdf"

candidate_paths = [
    Path(PDF_NAME),
    Path("data") / PDF_NAME,
    Path("/mnt/data") / PDF_NAME,  # ChatGPT 작업환경용 경로
]

PDF_PATH = next((path for path in candidate_paths if path.exists()), None)

if PDF_PATH is None:
    raise FileNotFoundError(
        f"PDF 파일을 찾을 수 없습니다. 다음 위치 중 하나에 파일을 넣어주세요: {candidate_paths}"
    )

print("PDF_PATH:", PDF_PATH)

PDF_PATH: data/진화하는 금융사기, 모두의 경계가 필요한 시점.pdf


## 4. PDF 로딩

PDF를 페이지 단위 `Document`로 로딩한다.  
각 페이지는 `page_content`와 `metadata`를 가진다.

### pdf 파일 로딩

In [5]:
# PDF 로드
loader = PyPDFLoader(str(PDF_PATH))

# PDF 페이지를 로드하여 문서 객체 리스트로 반환
pages = loader.load()

# 메타데이터 보강
for doc in pages:
    doc.metadata["source_name"] = PDF_NAME
    doc.metadata["page_label"] = doc.metadata.get("page", 0) + 1

print("로드된 페이지 수:", len(pages))
print("첫 번째 페이지 메타데이터:")
pprint(pages[0].metadata)
print("\n첫 번째 페이지 내용 일부:")
print(pages[0].page_content[:500])

로드된 페이지 수: 15
첫 번째 페이지 메타데이터:
{'author': 'user',
 'creationdate': '2026-05-29T09:03:29+09:00',
 'creator': 'Hancom PDF 1.3.0.534',
 'moddate': '2026-05-29T09:03:29+09:00',
 'page': 0,
 'page_label': 1,
 'pdfversion': '1.4',
 'producer': 'Hancom PDF 1.3.0.534',
 'source': 'data/진화하는 금융사기, 모두의 경계가 필요한 시점.pdf',
 'source_name': '진화하는 금융사기, 모두의 경계가 필요한 시점.pdf',
 'total_pages': 15}

첫 번째 페이지 내용 일부:
2026년 5월 29일 제 17 호
진화하는 금융사기,
모두의 경계가 필요한 시점


## 5. 문서 청킹

RAG에서는 문서 전체를 한 번에 넣지 않고, 검색 가능한 단위로 나눈다.  
 RecursiveCharacterTextSplitter()을 사용한다.  
이 실습에서는 보고서형 PDF에 적합한 기본값으로 `chunk_size=800`, `chunk_overlap=120`을 사용한다.

- `chunk_size`: 하나의 문서 조각 최대 길이
- `chunk_overlap`: 앞뒤 청크가 겹치는 길이
- 목적: 문맥 손실을 줄이면서 검색 단위를 적절히 유지하는 것

In [6]:
# 텍스트 분할 설정
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=100,
    length_function=len,
)

# 페이지 단위로 로드된 문서를 청크 단위로 분할
chunks = text_splitter.split_documents(pages)

# 청크 단위로 메타데이터 보강
for i, doc in enumerate(chunks):
    doc.metadata["chunk_id"] = i
    doc.metadata["page_label"] = doc.metadata.get("page", 0) + 1
    doc.metadata["source_name"] = PDF_NAME

print("전체 청크 수:", len(chunks))
print("첫 번째 청크 메타데이터:")
pprint(chunks[0].metadata)
print("\n첫 번째 청크 내용:")
print(chunks[0].page_content[:800])

전체 청크 수: 15
첫 번째 청크 메타데이터:
{'author': 'user',
 'chunk_id': 0,
 'creationdate': '2026-05-29T09:03:29+09:00',
 'creator': 'Hancom PDF 1.3.0.534',
 'moddate': '2026-05-29T09:03:29+09:00',
 'page': 0,
 'page_label': 1,
 'pdfversion': '1.4',
 'producer': 'Hancom PDF 1.3.0.534',
 'source': 'data/진화하는 금융사기, 모두의 경계가 필요한 시점.pdf',
 'source_name': '진화하는 금융사기, 모두의 경계가 필요한 시점.pdf',
 'total_pages': 15}

첫 번째 청크 내용:
2026년 5월 29일 제 17 호
진화하는 금융사기,
모두의 경계가 필요한 시점


## 6. 청크 분포 확인

페이지별 청크 수를 확인한다.  
특정 페이지의 청크가 지나치게 많거나 적으면 청킹 전략을 조정해야 한다.

In [7]:
# 청크 정보를 DataFrame으로 정리하여 페이지별 청크 수와 샘플을 확인
chunk_df = pd.DataFrame([
    {
        "chunk_id": doc.metadata["chunk_id"],
        "page": doc.metadata.get("page_label"),
        "text_length": len(doc.page_content),
        "preview": doc.page_content[:80].replace("\n", " ")
    }
    for doc in chunks
])

print("페이지별 청크 수")
display(chunk_df.groupby("page").size().reset_index(name="chunk_count"))

print("청크 샘플")
display(chunk_df.head(10))

페이지별 청크 수


,page,chunk_count
0,1,1
1,2,1
2,3,1
3,4,1
4,5,1
5,6,1
6,7,1
7,8,1
8,9,1
9,10,1


청크 샘플


,chunk_id,page,text_length,preview
0,0,1,45,"2026년 5월 29일 제 17 호 진화하는 금융사기, 모두의 경계가 필요한 시점"
1,1,2,1073,하나은행 ...
2,2,3,926,Hana Bank Hana Institute of Finance 2 ∎ 규제 공백 ...
3,3,4,1114,하나은행 ...
4,4,5,1126,Hana Bank Hana Institute of Finance 4 2. 금융사기 ...
5,5,6,934,하나은행 ...
6,6,7,1355,Hana Bank Hana Institute of Finance 6 III. 금융사...
7,7,8,1443,하나은행 ...
8,8,9,845,Hana Bank Hana Institute of Finance 8 2. 금융회사 ...
9,9,10,1192,하나은행 ...


## 7. 임베딩 모델과 FAISS 벡터스토어 생성

여기서부터 인덱싱 단계이다.

```text
청크 텍스트 → 임베딩 벡터 → FAISS 인덱스 저장
```

FAISS는 벡터 간 유사도 검색을 빠르게 수행하기 위한 로컬 벡터 검색 라이브러리이다.  
이 노트북에서는 LangChain의 `FAISS.from_documents()`를 사용해 문서 청크와 임베딩 모델을 연결한다.

In [9]:
# 1. 임베딩 모델 초기화
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    chunk_size=1000,
)

# 2. FAISS 인덱스 저장 경로 설정
INDEX_DIR = Path("faiss_financial_fraud_pdf_index")
FAISS_INDEX_FILE = INDEX_DIR / "index.faiss"
FAISS_PKL_FILE = INDEX_DIR / "index.pkl"

# 3. 기존 인덱스가 있으면 로딩, 없으면 새로 생성
if FAISS_INDEX_FILE.exists() and FAISS_PKL_FILE.exists():
    print("기존 FAISS 인덱스를 로딩합니다.")

    # FAISS 벡터스토어 로딩 
    vectorstore = FAISS.load_local(str(INDEX_DIR), embeddings)

    print("FAISS 인덱스 로딩 완료:", INDEX_DIR.resolve())

else:
    print("기존 FAISS 인덱스가 없습니다.")
    print("청크 문서를 임베딩하여 새 FAISS 벡터스토어를 생성합니다.")

    # FAISS 벡터스토어 생성
    vectorstore = FAISS.from_documents(chunks, embeddings)

    INDEX_DIR.mkdir(parents=True, exist_ok=True)
    vectorstore.save_local(str(INDEX_DIR))

    print("FAISS 인덱스 생성 및 저장 완료:", INDEX_DIR.resolve())

# 4. 인덱스 상태 확인
print("저장된 문서 청크 수:", vectorstore.index.ntotal)


기존 FAISS 인덱스가 없습니다.
청크 문서를 임베딩하여 새 FAISS 벡터스토어를 생성합니다.
FAISS 인덱스 생성 및 저장 완료: /home/aiuser/rag_agent_ex/toy_pjt1/faiss_financial_fraud_pdf_index
저장된 문서 청크 수: 15


## 8. 기본 유사도 검색 테스트

먼저 LLM 답변 생성 없이, 검색 결과만 확인한다.  
이 단계가 중요하다. 검색이 잘못되면 답변도 잘못될 가능성이 높다.

`similarity_search_with_score()`의 score는 기본 FAISS 설정에서는 거리 기반 점수로 해석한다.  
일반적으로 **낮을수록 더 가까운 문서**로 볼 수 있다.

In [10]:
question = "금융사기는 어떤 단계로 발생하나요?"

results = vectorstore.similarity_search_with_score(question, k=5)

for rank, (doc, score) in enumerate(results, start=1):
    print(f"\n[{rank}] score={score:.4f} | page={doc.metadata.get('page_label')} | chunk={doc.metadata.get('chunk_id')}")
    print(doc.page_content[:500].replace("\n", " "))


[1] score=0.8664 | page=4 | chunk=3
하나은행                                                                                                                    하나금융연구소 33 I. 금융사기란?  1. 금융사기의 정의와 특징 ■ 금융사기는 사람을 속이거나 착각하게 하여 금전적 이득을 취하는 범죄 행위 ● 금융사기는 사기 수법과 수단에 따라 양상이 다르게 나타나나, 공통적으로 ‘신뢰  유도-개인정보 수집-금전 편취’ 단계에 걸쳐 금전적 피해를 유발 - 택배회사 발송 배송조회 QR코드(신뢰)로 위장하여 피해자가 코드 스캔 시 악성코드나  앱을 스마트폰에 설치(수집)하고, 이를 통해 탈취한 개인정보로 금전을 유출(탈취) - 유명 핀플루언서(금융+인플루언서)의 영상을 도용(신뢰)하여 가짜 채널로 가입을 권유 (수집)한 후 미끼 수익을 제공(신뢰)하고 더 큰 금액의 투자를 유도(탈취)한 뒤 잠적 ■ 대면에 기반한 과거 사기 수법과는 달

[2] score=0.9186 | page=2 | chunk=1
하나은행                                                                                                                    하나금융연구소 11 ∎ 금융사기는 의도적으로 사람을 속이거나 착각하게 하여 금전적 이득을  취하는 범죄 행위로 일반적으로 전기통신금융사기와 투자사기로 구분 l 금융사기는 금전적 피해를 유발하는 범죄 행위 전반을 포괄해 다양한 기준으로  구분되나 전기통신금융사기와 투자사기로 구분하는 것이 보편적 l 사기 수법과 수단에 따라 양상이 다르게 나타나며, 공통적으로 ‘신뢰 유도-  개인정보 수집-금전 편취’ 단계를 거쳐 금전적 피해를 유발 - 택배회사에서 보낸 배송조회 큐알코드(신뢰)로 위장하여 피해자가 코드 스캔 시  악성코드나 앱을 스마트폰에 설

## 9. Retriever 구성

Retriever는 사용자의 질문을 받아 관련 문서 조각을 반환하는 객체이다.

```text
질문 → Retriever → 관련 청크 Top-k 반환
```

In [11]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 5})

retrieved_docs = retriever.invoke("전기통신금융사기와 투자사기의 차이는 무엇인가요?")

for doc in retrieved_docs:
    print(f"page={doc.metadata.get('page_label')}, chunk={doc.metadata.get('chunk_id')}")
    print(doc.page_content[:300].replace("\n", " "))
    print("-" * 80)

page=5, chunk=4
Hana Bank Hana Institute of Finance 4 2. 금융사기 주요 유형 ■ 금융사기는 금전적 피해를 유발하는 범죄행위 전반을 포괄해 다양한 기준으로  구분되나 전기통신금융사기와 투자사기 두 분류로 구분하는 것이 보편적 ● 전기통신금융사기는 전화, 메신저, 인터넷 등 전기통신을 이용하여 금융기관·    지인·가족 등을 사칭하거나 허위 사실로 피해자의 자금을 탈취하는 수법 - 피싱, 스미싱, 파밍 등이 대표적인 유형이며, 법에서 열거하는 통신수단을 이용하여   개인정보를 노출시켜 자금을 송금·이체하도록 유도 - ’
--------------------------------------------------------------------------------
page=7, chunk=6
Hana Bank Hana Institute of Finance 6 III. 금융사기 대응을 위한 노력 1. 규제 공백 해소를 위한 제도 보완 ■ 신규 사기 수법의 등장으로 발생하는 규제 공백을 해소하고자 법을 제정하거나  기존 법을 개정하여 책임 소재 기준, 피해 보상 체계 등을 지속적으로 정비 ● ’24.1월 자본시장법을 개정하여 불공정거래행위에 대한 과징금을 신설하고,  부당이득에 관한 산정 기준을 명확히 하여 법적 실효성을 제고 - 3대 불공정거래행위(시세조종, 미공개정보 이용행위, 부정거래)에 대해 부당이득의    최대 2
--------------------------------------------------------------------------------
page=2, chunk=1
하나은행                                                                                                                    하나금융연구소 11 ∎ 금융사기는 의도적으로 사람을 속이거나 착각하게 하여 금전적 이득을  취하는 범죄 행위로 일반적으로 

## 10. RAG 답변 프롬프트 구성

핵심은 LLM이 검색된 문서 밖의 내용을 임의로 생성하지 않도록 제약하는 것이다.

프롬프트 설계 원칙은 다음과 같다.

1. 문서 근거만 사용한다.
2. 문서에서 확인할 수 없으면 모른다고 답한다.
3. 답변 마지막에 근거 페이지를 포함한다.

In [12]:
# 검색 chunk를 context로 변환하는 함수
def format_docs(docs):
    """검색된 문서들을 LLM 입력용 context 문자열로 변환한다."""
    formatted = []
    for doc in docs:
        page = doc.metadata.get("page_label", "?")
        chunk_id = doc.metadata.get("chunk_id", "?")
        formatted.append(
            f"[출처: page {page}, chunk {chunk_id}]\n{doc.page_content}"
        )
    return "\n\n".join(formatted)


# 4. LLM과 프롬프트 초기화
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
당신은 금융사기 보고서 기반 QA 어시스턴트입니다.
아래 [문서]에 근거해서만 답변하세요.
문서에서 확인할 수 없는 내용은 '문서에서 확인할 수 없습니다.'라고 답변하세요.
답변은 핵심 위주로 3~5문장으로 정리하세요.
답변 마지막에 근거 페이지를 '근거: p.숫자' 형식으로 표시하세요.

[문서]
{context}
""".strip()
    ),

    ("human", "{question}"),
])

# model = gpt-4o-mini로 설정, temperature=0으로 설정
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# 5. LCEL Chain 구성(프롬프트, LLM, 문자열 출력 파서)
answer_chain = prompt | llm | StrOutputParser()

In [13]:
question = "전기통신금융사기와 투자사기는 어떻게 다른가요?"

# 질문에 대한 유사도 top 4 검색 하기
docs = vectorstore.similarity_search(question, k=4)
# 검색된 문서를 LLM 입력용 context로 변환
context = format_docs(docs)

# LLM에 질문과 검색된 문서 컨텍스트를 입력하여 답변 생성
answer = prompt | llm | StrOutputParser()

In [14]:
# 결과 출력
answer

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="당신은 금융사기 보고서 기반 QA 어시스턴트입니다.\n아래 [문서]에 근거해서만 답변하세요.\n문서에서 확인할 수 없는 내용은 '문서에서 확인할 수 없습니다.'라고 답변하세요.\n답변은 핵심 위주로 3~5문장으로 정리하세요.\n답변 마지막에 근거 페이지를 '근거: p.숫자' 형식으로 표시하세요.\n\n[문서]\n{context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})])
| ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.4.3', 'langchain-openai': '1.6.6'}}, profile={'name': 'GPT-4.1 mini', 'release_date': '2025-04-14', 'last_updated': '2025-04-14', 'open_weights': False, 'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inpu

## 11. RAG 함수 만들기

검색과 답변 생성을 하나의 함수로 묶는다.

```text
질문 입력
→ FAISS 검색
→ 검색 문서 format
→ LLM 답변 생성
→ 답변과 출처 반환
```

In [15]:
def answer_question(question: str, k: int = 4):
    """질문에 대해 FAISS 검색 후 문서 기반 답변과 출처 테이블을 반환한다."""
    docs = vectorstore.similarity_search(question, k=k)
    context = format_docs(docs)

    answer = answer_chain.invoke({
        "question": question,
        "context": context,
    })

    sources = pd.DataFrame([
        {
            "rank": i + 1,
            "page": doc.metadata.get("page_label"),
            "chunk_id": doc.metadata.get("chunk_id"),
            "preview": doc.page_content[:120].replace("\n", " ")
        }
        for i, doc in enumerate(docs)
    ])

    return answer, sources

## 12. RAG 실행 예시

다음 질문들은 이 보고서에 포함된 내용을 기반으로 답변 가능한 질문이다.

In [16]:
question = "전기통신금융사기와 투자사기는 어떻게 다른가요?"

answer, sources = answer_question(question, k=4)

print("[질문]")
print(question)
print("\n[답변]")
print(answer)

print("\n[검색된 근거]")
display(sources)  # Jupyter 환경에서는 display로 테이블이 예쁘게 나옵니다.


[질문]
전기통신금융사기와 투자사기는 어떻게 다른가요?

[답변]
전기통신금융사기는 전화, 메신저, 인터넷 등 전기통신을 이용해 금융기관이나 지인 등을 사칭하거나 허위 사실로 피해자의 자금을 탈취하는 수법으로, 피싱, 스미싱, 파밍 등이 대표적입니다. 반면 투자사기는 투자자의 수익 욕구를 이용해 투자를 유도하거나 가격을 조작해 금전적 피해를 입히는 범죄로, 리딩방 사기, 비상장 주식 과장 판매, 가짜 HTS/MTS 이용 등이 있습니다. 전기통신금융사기는 비대면 통신 수단을 주로 활용하는 반면, 투자사기는 과거 대면 기반이었으나 최근 SNS와 미디어를 통한 수법으로 변화하고 있습니다. 두 유형 모두 법적 규제와 대응 방안이 다르게 마련되고 있습니다.  
근거: p.2, p.5

[검색된 근거]


,rank,page,chunk_id,preview
0,1,5,4,Hana Bank Hana Institute of Finance 4 2. 금융사기 ...
1,2,7,6,Hana Bank Hana Institute of Finance 6 III. 금융사...
2,3,3,2,Hana Bank Hana Institute of Finance 2 ∎ 규제 공백 ...
3,4,2,1,하나은행 ...


In [17]:
question = "금융회사들은 금융사기를 예방하기 위해 어떤 시스템을 도입하고 있나요?"

answer, sources = answer_question(question, k=4)

print("[질문]")
print(question)
print("\n[답변]")
print(answer)
print("\n[검색된 근거]")
display(sources)

[질문]
금융회사들은 금융사기를 예방하기 위해 어떤 시스템을 도입하고 있나요?

[답변]
금융회사들은 이상금융거래탐지시스템(Fraud Detection System)을 도입하여 거래 내역, 고객 정보, 거래 패턴 등을 실시간 분석해 부정 거래를 탐지하고 차단하고 있습니다. 또한 AI 기술을 접목해 신종 사기 수법에 선제적으로 대응하며, 금융보안원, 통신사 등 관련 기관과 협업해 실시간 정보 공유 체계를 구축하고 있습니다. 해외 사례로는 미국 JPMorgan Chase의 AI 기반 ‘NeuroShield’, 호주 Commonwealth Bank의 ‘Scam Indicator’ 등 다양한 탐지 및 예방 기술도 도입 중입니다. 금융사기 전담 조직 확충과 자체 사기 방지 소프트웨어 개발도 병행하고 있습니다. 근거: p.9, p.12

[검색된 근거]


,rank,page,chunk_id,preview
0,1,9,8,Hana Bank Hana Institute of Finance 8 2. 금융회사 ...
1,2,12,11,하나은행 ...
2,3,13,12,Hana Bank Hana Institute of Finance 12 V. 대응방안...
3,4,3,2,Hana Bank Hana Institute of Finance 2 ∎ 규제 공백 ...


## 13. 문서에 없는 질문 테스트

RAG 실습에서는 답변 가능한 질문만 넣으면 안 된다.  
문서에 없는 질문을 넣어 **모른다고 답하는지** 확인해야 한다.

In [18]:
question = "2026년 6월 기준 최신 보이스피싱 피해 금액은 얼마인가요?"

answer, sources = answer_question(question, k=4)

print("[질문]")
print(question)
print("\n[답변]")
print(answer)
print("\n[검색된 근거]")
display(sources)

[질문]
2026년 6월 기준 최신 보이스피싱 피해 금액은 얼마인가요?

[답변]
문서에 따르면 2024년 기준 보이스피싱 등 불특정 다수를 대상으로 하는 금융사기 피해 금액은 약 1.7조 원이며, 2025년 피해액도 증가 추세임이 언급되어 있습니다. 그러나 2026년 6월 기준 최신 보이스피싱 피해 금액에 대한 구체적인 수치는 문서에 포함되어 있지 않습니다. 따라서 2026년 6월 기준 최신 피해 금액은 문서에서 확인할 수 없습니다. 

근거: p.6

[검색된 근거]


,rank,page,chunk_id,preview
0,1,10,9,하나은행 ...
1,2,6,5,하나은행 ...
2,3,8,7,하나은행 ...
3,4,1,0,"2026년 5월 29일 제 17 호 진화하는 금융사기, 모두의 경계가 필요한 시점"


## 14. 정리

이 실습에서 구현한 Naive RAG 흐름은 다음과 같다.

```text
PDF 로딩
→ 페이지 단위 Document 생성
→ RecursiveCharacterTextSplitter로 청킹
→ OpenAIEmbeddings로 임베딩
→ FAISS 벡터스토어 생성
→ 질문 기반 Top-k 검색
→ 검색 문서 기반 LLM 답변 생성
```

## 핵심 포인

| 관찰 포인트 | 핵심 내용 |
|---|---|
| 검색 결과가 부정확함 | 청크 크기, 질문 표현, Top-k 조정 필요 |
| 답변이 문서 밖 내용을 포함함 | 프롬프트 제약 및 문서 없음 질문 테스트 필요 |
| 표·그림 내용이 누락됨 | PDF 파싱 방식의 한계, Markdown 변환 비교 필요 |
| 같은 질문인데 답변이 달라짐 | 검색 결과와 LLM 생성 조건을 분리해서 점검해야 함 |
| 인덱스 재사용 필요 | FAISS 저장·로드 구조로 인덱싱 비용 절감 가능 |